# Homework 4 - Computing Point-in-Time Residual Returns
In this homework, we will use regressions to compute beta-adjusted "residual" returns in a point-in-time fashion suitable for backtesting / live trading.


1. Download Daily Bars for FB, AAPL, AMZN, NFLX, GOOGL and QQQ from yahoo finance starting 2016-01-01. Use the Adj Close to compute daily returns.
2. Now, let's compute the beta of FB, AAPL, AMZN, NFLX, GOOGL using QQQ as our benchmark. You can think of this as the beta these stocks have to their industry (tech). In practice,  we have to use some lookback window to compute the beta. Let's use 252 (1 year, excluding wknds/holidays). So, for each day, the betas should be computed using the most recent 252 data points.
3. Using the betas, compute an "alpha" on each day. This is also known as a "residual return".
4. Compare the volatility of the residual returns to that of the original returns. What do you notice?
5. Compare the pairwise correlations of the residual returns to that of the original returns. What do you notice?
6. Compute the information ratio for each of these stocks and compare that to the sharpe ratio.


In [2]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
import yfinance as yf 

In [3]:
univ = ['META','AAPL','AMZN','NFLX','GOOGL','QQQ']

stock_px = yf.download(univ, start="2016-01-01")

[*********************100%***********************]  6 of 6 completed


In [4]:
stock_px

Price            Close                                                 \
Ticker            AAPL        AMZN       GOOGL        META       NFLX   
Date                                                                    
2016-01-04   23.709099   31.849501   37.638256  101.330162  10.996000   
2016-01-05   23.114967   31.689501   37.741840  101.835724  10.766000   
2016-01-06   22.662611   31.632500   37.632805  102.073631  11.768000   
2016-01-07   21.706154   30.396999   36.724361   97.067589  11.456000   
2016-01-08   21.820927   30.352501   36.224289   96.482727  11.139000   
...                ...         ...         ...         ...        ...   
2026-06-12  291.130005  238.550003  359.679993  566.454956  80.339996   
2026-06-15  296.420013  246.020004  369.350006  593.479980  81.669998   
2026-06-16  299.239990  246.000000  373.250000  600.210022  78.720001   
2026-06-17  295.950012  237.500000  363.790009  567.580017  76.959999   
2026-06-18  298.010010  244.389999  368.029999  577.219971  77.379997   

Price                         High                                      ...  \
Ticker             QQQ        AAPL        AMZN       GOOGL        META  ...   
Date                                                                    ...   
2016-01-04  101.717300   23.713601   32.886002   37.775044  101.349985  ...   
2016-01-05  101.540794   23.821627   32.345501   38.121969  102.807189  ...   
2016-01-06  100.565422   23.038447   31.989500   37.949992  102.866663  ...   
2016-01-07   97.416367   22.534341   31.500000   37.433573  100.547036  ...   
2016-01-08   96.617447   22.304786   31.207001   37.176346   99.625130  ...   
...                ...         ...         ...         ...         ...  ...   
2026-06-12  721.340027  297.140015  243.360001  366.570007  575.536566  ...   
2026-06-15  744.000000  297.779999  247.809998  372.989990  601.270020  ...   
2026-06-16  729.859985  300.480011  249.509995  376.000000  605.809998  ...   
2026-06-17  722.510010  302.070007  245.910004  372.329987  593.809998  ...   
2026-06-18  740.619995  300.570007  245.729996  369.480011  580.219971  ...   

Price             Open                                        Volume  \
Ticker           GOOGL        META       NFLX         QQQ       AAPL   
Date                                                                   
2016-01-04   37.775044  101.062508  10.900000  101.670851  270597600   
2016-01-05   37.869213  101.994327  11.045000  102.218913  223164000   
2016-01-06   37.188744  100.249645  10.529000   99.775837  273829600   
2016-01-07   36.996451   99.625131  11.636000   98.419600  324377600   
2016-01-08   37.061364   99.010524  11.633000   98.122300  283192000   
...                ...         ...        ...         ...        ...   
2026-06-12  362.619995  572.419460  81.580002  717.609985   38742100   
2026-06-15  367.929993  579.900024  80.620003  738.099976   45732600   
2026-06-16  369.600006  593.650024  81.900002  742.250000   39874400   
2026-06-17  369.130005  592.000000  78.099998  735.190002   42745100   
2026-06-18  365.750000  572.820007  76.930000  737.200012   85962200   

Price                                                           
Ticker           AMZN     GOOGL      META       NFLX       QQQ  
Date                                                            
2016-01-04  186290000  67382000  37912400  207948000  50807600  
2016-01-05  116452000  45216000  23258200  176646000  38795200  
2016-01-06  106584000  48206000  25096200  330457000  41891100  
2016-01-07  141498000  63132000  45172900  336367000  61386300  
2016-01-08  110258000  47506000  35402300  180671000  69344000  
...               ...       ...       ...        ...       ...  
2026-06-12   51186600  24704600  14326400   35309700  51168400  
2026-06-15   41711600  27727500  17653300   36430800  46710200  
2026-06-16   35187400  24876100  11344400   65054000  45348700  
2026-06-17   44780800  24543600  20478300   50338800  51669300  
2026-06-18   756

In [5]:
daily_returns = stock_px['Close'] / stock_px['Close'].shift(1) -1


In [6]:
daily_returns

# Download Daily Bars for FB, AAPL, AMZN, NFLX, GOOGL and QQQ from yahoo finance starting 2016-01-01. 
# Use the Adj Close to compute daily returns.
# have been done

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Date,,,,,,
2016-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-05,-0.025059,-0.005024,0.002752,0.004989,-0.020917,-0.001735
2016-01-06,-0.019570,-0.001799,-0.002889,0.002336,0.093071,-0.009606
2016-01-07,-0.042204,-0.039058,-0.024140,-0.049043,-0.026513,-0.031314
2016-01-08,0.005288,-0.001464,-0.013617,-0.006025,-0.027671,-0.008201
...,...,...,...,...,...,...
2026-06-12,-0.015222,-0.012256,0.005339,-0.002551,-0.011443,0.005885
2026-06-15,0.018171,0.031314,0.026885,0.047709,0.016555,0.031414
2026-06-16,0.009513,-0.000081,0.010559,0.011340,-0.036121,-0.019005


In [7]:
# Now, let's compute the beta of FB, AAPL, AMZN, NFLX, GOOGL using QQQ as our benchmark. 
# You can think of this as the beta these stocks have to their industry (tech). 
# In practice, we have to use some lookback window to compute the beta. Let's use 252 (1 year, excluding wknds/holidays). 
# So, for each day, the betas should be computed using the most recent 252 data points.

corr = daily_returns.rolling(252).corr(daily_returns['QQQ'])
vol = daily_returns.rolling(252).std()
beta = (corr*vol).divide(vol['QQQ'], axis=0)

In [8]:
###Using the betas, compute an "alpha" on each day. This is also known as a "residual return".
resid = daily_returns - beta.multiply(daily_returns['QQQ'],0)

In [9]:
# Compare the volatility of the residual returns to that of the original returns. 
# What do you notice?

vol = {}
vol['orig'] = daily_returns.std()*np.sqrt(252)
vol['resid'] = resid.std()*np.sqrt(252)
vol = pd.DataFrame(vol).drop('QQQ')
vol

,orig,resid
Ticker,,
AAPL,0.288552,0.172885
AMZN,0.327187,0.206369
GOOGL,0.289101,0.187182
META,0.386189,0.280542
NFLX,0.417474,0.330353


In [11]:
daily_returns

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Date,,,,,,
2016-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-05,-0.025059,-0.005024,0.002752,0.004989,-0.020917,-0.001735
2016-01-06,-0.019570,-0.001799,-0.002889,0.002336,0.093071,-0.009606
2016-01-07,-0.042204,-0.039058,-0.024140,-0.049043,-0.026513,-0.031314
2016-01-08,0.005288,-0.001464,-0.013617,-0.006025,-0.027671,-0.008201
...,...,...,...,...,...,...
2026-06-12,-0.015222,-0.012256,0.005339,-0.002551,-0.011443,0.005885
2026-06-15,0.018171,0.031314,0.026885,0.047709,0.016555,0.031414
2026-06-16,0.009513,-0.000081,0.010559,0.011340,-0.036121,-0.019005


In [12]:
resid

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Date,,,,,,
2016-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-05,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-06,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-07,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-08,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
2026-06-12,-0.018667,-0.018323,0.000111,-0.008566,-0.013146,-5.117434e-17
2026-06-15,-0.000185,-0.001063,-0.000945,0.015054,0.007176,-2.636780e-16
2026-06-16,0.020305,0.019323,0.027103,0.030677,-0.030056,1.595946e-16


In [13]:
###Compare the pairwise correlations of the residual returns to that of the original returns. 
# What do you notice?
daily_returns.corr()


Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,0.564970,0.596013,0.511651,0.415130,0.794124
AMZN,0.564970,1.000000,0.632731,0.605437,0.514029,0.761766
GOOGL,0.596013,0.632731,1.000000,0.598624,0.429913,0.773830
META,0.511651,0.605437,0.598624,1.000000,0.442822,0.692554
NFLX,0.415130,0.514029,0.429913,0.442822,1.000000,0.568450
QQQ,0.794124,0.761766,0.773830,0.692554,0.568450,1.000000


In [14]:
resid.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,-0.084546,-0.052323,-0.076909,-0.092646,0.045777
AMZN,-0.084546,1.000000,0.086007,0.141149,0.139855,0.005166
GOOGL,-0.052323,0.086007,1.000000,0.121748,-0.047764,0.025854
META,-0.076909,0.141149,0.121748,1.000000,0.078925,0.015026
NFLX,-0.092646,0.139855,-0.047764,0.078925,1.000000,0.017931
QQQ,0.045777,0.005166,0.025854,0.015026,0.017931,1.000000


In [15]:
# Compute the information ratio for each of these stocks and compare that to the sharpe ratio.
df = {}
df['IR'] = resid.mean()/resid.std() * np.sqrt(252)
df['SR'] = daily_returns.mean()/daily_returns.std()*np.sqrt(252)
df = pd.DataFrame(df).drop('QQQ')
df


,IR,SR
Ticker,,
AAPL,0.341028,0.985298
AMZN,-0.041566,0.760419
GOOGL,0.254342,0.900765
META,-0.074447,0.626840
NFLX,0.141806,0.660914
